<a href="https://colab.research.google.com/github/RaphaelRAY/projeto-bi-games/blob/main/notebooks/02_criar_view.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2. Criação da View de Análise no BigQuery
Este notebook consolida as tabelas importadas em uma única visão (View) em um **dataset separado** (`games_analytics`), seguindo boas práticas de arquitetura de dados.

In [ ]:
import pandas as pd
from google.cloud import bigquery
from google.colab import auth

# Autenticação no Google Colab
auth.authenticate_user()

In [ ]:
project_id = 'directed-mender-489100-m3'
dataset_raw = 'games_data'           # Onde estão os CSVs
dataset_analytics = 'games_analytics' # Onde ficará a View final

client = bigquery.Client(project=project_id)

In [15]:
# Garante que o dataset de analytics existe
dataset_ref = client.dataset(dataset_analytics)
try:
    client.get_dataset(dataset_ref)
    print(f"Dataset {dataset_analytics} já existe.")
except Exception:
    print(f"Criando dataset {dataset_analytics}...")
    client.create_dataset(bigquery.Dataset(dataset_ref))

# 1. View Detalhada (Para gráficos de barras e filtros)
view_id = f"{project_id}.{dataset_analytics}.vw_analise_games"

sql_view = f"""
CREATE OR REPLACE VIEW `{view_id}` AS
SELECT
    g.id AS game_id,
    g.name AS nome_jogo,
    g.released AS data_lancamento,
    g.rating AS nota_usuarios,
    g.metacritic AS nota_critica,
    g.playtime AS tempo_jogo_horas,
    p.publisher_name AS distribuidora,
    gen.genre_name AS genero,
    plat.platform_name AS plataforma,
    m.completability_index AS indice_conclusao,
    m.consensus_score AS score_consenso,
    m.time_rating_ratio AS relacao_tempo_nota
FROM
    `{project_id}.{dataset_raw}.games` g
LEFT JOIN
    `{project_id}.{dataset_raw}.game_publishers` p ON CAST(g.id AS STRING) = CAST(p.game_id AS STRING)
LEFT JOIN
    `{project_id}.{dataset_raw}.game_genres` gen ON CAST(g.id AS STRING) = CAST(gen.game_id AS STRING)
LEFT JOIN
    `{project_id}.{dataset_raw}.game_platforms` plat ON CAST(g.id AS STRING) = CAST(plat.game_id AS STRING)
LEFT JOIN
    `{project_id}.{dataset_raw}.game_derived_metrics` m ON CAST(g.id AS STRING) = CAST(m.game_id AS STRING)
"""

# 2. View Unificada (Para KPIs e Médias sem duplicatas)
view_unica_id = f"{project_id}.{dataset_analytics}.vw_analise_games_unica"

sql_view_unica = f"""
CREATE OR REPLACE VIEW `{view_unica_id}` AS
SELECT
    g.id AS game_id,
    g.name AS nome_jogo,
    g.released AS data_lancamento,
    g.rating AS nota_usuarios,
    g.metacritic AS nota_critica,
    g.playtime AS tempo_jogo_horas,
    STRING_AGG(DISTINCT p.publisher_name, ', ') AS distribuidoras,
    STRING_AGG(DISTINCT plat.platform_name, ', ') AS plataformas,
    STRING_AGG(DISTINCT gen.genre_name, ', ') AS generos,
    AVG(m.completability_index) AS indice_conclusao,
    AVG(m.consensus_score) AS score_consenso,
    AVG(m.time_rating_ratio) AS relacao_tempo_nota
FROM
    `{project_id}.{dataset_raw}.games` g
LEFT JOIN `{project_id}.{dataset_raw}.game_publishers` p ON CAST(g.id AS STRING) = CAST(p.game_id AS STRING)
LEFT JOIN `{project_id}.{dataset_raw}.game_platforms` plat ON CAST(g.id AS STRING) = CAST(plat.game_id AS STRING)
LEFT JOIN `{project_id}.{dataset_raw}.game_genres` gen ON CAST(g.id AS STRING) = CAST(gen.game_id AS STRING)
LEFT JOIN `{project_id}.{dataset_raw}.game_derived_metrics` m ON CAST(g.id AS STRING) = CAST(m.game_id AS STRING)
GROUP BY 1, 2, 3, 4, 5, 6
"""

print(f"Criando/Atualizando views no dataset {dataset_analytics}...")
client.query(sql_view).result()
client.query(sql_view_unica).result()
print(f"✅ Sucesso! Views criadas/atualizadas:\n - {view_id}\n - {view_unica_id}")


Dataset games_analytics já existe.
Criando/Atualizando views no dataset games_analytics...
✅ Sucesso! Views criadas/atualizadas:
 - directed-mender-489100-m3.games_analytics.vw_analise_games
 - directed-mender-489100-m3.games_analytics.vw_analise_games_unica
